In [1]:
import numpy as np
import csv,os
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity

#import data from CSVs
data = []
files=["../2_Database/TopBeerData.csv","../2_Database/GoodBeerData.csv","../2_Database/RestBeerData.csv"]
for i in range (0,len(files)):
    with open(files[i],"r",encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader) # remove header
        for row in reader:
            data.append(row)
print(len(data))
with open("BeerData.csv", 'w', newline='',encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['beer_text','brewery','beer'])
textual_data = []
with open("BeerData.csv", "a", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for row in data[1:]:
        beer_text = (
#            f"brewery: {row[2]}\n" #brewery removed since adds bias
#            f"beer name: {row[3]} |"
            f"beer kind: {row[4]} |"
            f"description: {row[11]}|"
            f"ABV: {row[6]} |"
            f"IBU: {row[7]} |"
            f"rating: {row[8]}"
        )
        writer.writerow([beer_text,row[2],row[3]])
        textual_data.append(beer_text)
print("Adatok kiíratva CSV-be.")

16405
Adatok kiíratva CSV-be.


In [2]:
# ---------------------------------------------------
# generate embeddings with OpenAI - COSTS MONEY
# ---------------------------------------------------

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")
)

embeddings = []
batch_size = 100
for i in range(0, len(textual_data), batch_size):
    batch = textual_data[i:i + batch_size]
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )
    embeddings.extend(
        item.embedding
        for item in response.data
    )

print(len(embeddings))

X_emb = np.array(embeddings)
print(X_emb.shape)

with open("BeerEmbeddings.csv", 'w', newline='',encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([""] * 1536) #1536 nr of coloumns  
    for row in X_emb:
        writer.writerow(row)
print("Embedding kiíratva CSVbe.")

16404
(16404, 1536)
Embedding kiíratva CSVbe.


In [3]:
#embeddings to be generated inside of multiple CSVs to be able to check in into GIT
#current size of embedding is 16399*1536 --> 450MB that means 1500x1536 will be something about 50MB which is okay
import pandas as pd
import math
import os
import sys

INPUT_FILE = "BeerEmbeddings.csv"
CHUNK_SIZE = 1500

try:
    df = pd.read_csv(INPUT_FILE)
except FileNotFoundError:
    print("Source file is not located in the directory!")
    sys.exit()
finally:
    pass # do nothing

num_chunks = math.ceil(len(df) / CHUNK_SIZE)

for i in range(num_chunks):
    chunk = df.iloc[i*CHUNK_SIZE:(i+1)*CHUNK_SIZE]

    chunk.to_csv(
        f"BeerEmbeddings{i+1}.csv",
        index=False
    )

    print(f"BeerEmbeddings{i+1}.csv elkészült ({len(chunk)} sor)")

try:
    os.remove(INPUT_FILE)
except FileNotFoundError:
    print("Source file is not located anymore in the directory!")
finally:
    print("Source file removed and outputs are created!")

BeerEmbeddings1.csv elkészült (1500 sor)
BeerEmbeddings2.csv elkészült (1500 sor)
BeerEmbeddings3.csv elkészült (1500 sor)
BeerEmbeddings4.csv elkészült (1500 sor)
BeerEmbeddings5.csv elkészült (1500 sor)
BeerEmbeddings6.csv elkészült (1500 sor)
BeerEmbeddings7.csv elkészült (1500 sor)
BeerEmbeddings8.csv elkészült (1500 sor)
BeerEmbeddings9.csv elkészült (1500 sor)
BeerEmbeddings10.csv elkészült (1500 sor)
BeerEmbeddings11.csv elkészült (1404 sor)
Source file removed and outputs are created!


In [241]:
from difflib import SequenceMatcher

beer_data=[]
with open("BeerData.csv","r",encoding="utf-8") as file:
    reader = csv.reader(file)
    next(reader) # remove header
    for row in reader:
        beer_data.append(row)

#get user beer preference
pref_beer = input("Add meg a sör nevét: ")

best_sim=0
count=0
for i, text in enumerate(beer_data):
    if pref_beer.lower() in text[2].lower():
        print(i,text[2],"Full matching of the beer name!!! \n")
        count=1
        best_sim=1
        best_idx = i
        best_text=text[2]
        break
    else:
        sim = SequenceMatcher(None, text[2].lower(), pref_beer.lower()).ratio()  
        if sim > best_sim:
            best_sim = sim
            best_idx = i
            best_text=text[2]
    
if (count==0 and best_sim<0.5) :     #0.5 is arbitrary
    print("nem találtam a kiválasztott sört")
else:
    print(f"Index: {best_idx}")
    print(f"Similarity: {best_sim:.3f}")
    print(f"A kiválasztott sör neve: {best_text}")
    pref_beer=best_text

Add meg a sör nevét:  Bagoly


217 Bagoly BA - Gemenc Whiskey Full matching of the beer name!!! 

Index: 217
Similarity: 1.000
A kiválasztott sör neve: Bagoly BA - Gemenc Whiskey


In [239]:
#CSV reading
emb_data=[]
with open("BeerEmbeddings.csv","r",encoding="utf-8") as file:
    reader = csv.reader(file)
    next(reader) # remove header
    for row in reader:
        emb_data.append(row)
    
    #cosine similarity
    emb_data = np.array(emb_data, dtype=np.float32) #necessary conversion
    
    pref = emb_data[best_idx].reshape(1, -1)
    similarities = cosine_similarity(pref, emb_data)[0]
    top_k = 5
    top_idx = np.argsort(similarities)[::-1] 
    top_idx = top_idx[top_idx != (best_idx)][:top_k]
    print(top_idx,"\n")
    for i in top_idx:
        print(i, similarities[i],beer_data[i])
        print("\n")


[ 227  226  217 1589  218] 

227 0.9183923 ['beer name: Birodalmi Bagoly - Kentucky Straight Bourbon Whisky BA 2023 |beer kind: Porter - Imperial / Double Baltic |description: Kentucky Straight Bourbon Whisky hordókban érlelt Birodalmi Bagoly 2023-as verziója. ez most 22 hónapot kapott.//2023 version of the Kentucky Straight Bourbon Whisky barrel aged Birodalmi Bagoly. this one was sentenced for 22 months. |ABV: 12.0 |IBU: N/A |rating: 3.9', 'Balkezes', 'Birodalmi Bagoly - Kentucky Straight Bourbon Whisky BA 2023']


226 0.880522 ['beer name: Birodalmi Bagoly - Rhum Agricole BA |beer kind: Porter - Imperial / Double Baltic |description: imperial baltic porter a Bagoly hagyományait továbbörökítve, rhum agricole hordóban élelve. |ABV: 12.0 |IBU: N/A |rating: 4.04', 'Balkezes', 'Birodalmi Bagoly - Rhum Agricole BA']


217 0.8531274 ['beer name: Bagoly BA - Gemenc Whiskey |beer kind: Porter - Baltic |description: Bagoly balti porter, Gemenc Whiskey hordókban érlelve. |ABV: 8.8 |IBU: 50 |ra